# Création du corpus annoté en `.conllu` **rempli** à partir de Propp

Ce notebook reconstruit un `.conllu` **à partir des sorties Propp**, sans relancer spaCy dans le notebook.

Il exploite directement le fichier `*.tokens`, qui contient déjà :
- `word`
- `lemma`
- `POS_tag`
- `dependency_relation`
- `syntactic_head_ID`

Il convertit ensuite :
- `token_ID_within_sentence` → `ID` CoNLL-U (local à la phrase, en **1-based**)
- `syntactic_head_ID` (ID document) → `HEAD` CoNLL-U (local à la phrase)
- `ROOT` → `HEAD = 0`

Les colonnes 9 et 10 restent :
- colonne 9 : annotation entité ou Dynamouv
- colonne 10 : type entité ou `"VERB"`

Le format de sortie est donc :
`ID  FORM  LEMMA  UPOS  XPOS  FEATS  HEAD  DEPREL  COL8  COL9`


In [1]:
import os
import csv
import re
from pathlib import Path
from collections import defaultdict


## 1) Paramètres

Adaptez si nécessaire :
- `inputDir` : dossier contenant `*.tokens` et `*.entities`
- `outputDir` : dossier de sortie des `.conllu`
- `dynamouv_csv` : chemin vers `dynaMouv.csv`


In [ ]:
inputDir = "../data/SACR"
outputDir = "../results/conllu/from_sacr"
dynamouv_csv = "../data/dynaMouv.csv"

Path(outputDir).mkdir(parents=True, exist_ok=True)

print("inputDir :", Path(inputDir).resolve())
print("outputDir:", Path(outputDir).resolve())
print("dynamouv :", Path(dynamouv_csv).resolve(), "(exists =", Path(dynamouv_csv).exists(), ")")


inputDir : C:\Users\arnod\stage_lattice\Model_TDM\data\PROPP\new_txt
outputDir: C:\Users\arnod\stage_lattice\Model_TDM\results\conllu\from_propp
dynamouv : C:\Users\arnod\stage_lattice\Model_TDM\data\dynaMouv.csv (exists = True )


## 2) Fonctions utilitaires

In [3]:
def normalize_for_dynamouv(tok: str) -> str:
    t = (tok or "").lower().strip()
    t = re.sub(r"^[\W_]+|[\W_]+$", "", t)
    return t


def detect_delimiter(path, encoding="utf-8"):
    path = Path(path)
    with path.open("r", encoding=encoding, newline="") as f:
        sample = f.read(4096)
    return "\t" if "\t" in sample else ","


def read_tokens_dict(tokens_path, encoding="utf-8"):
    tokens_path = Path(tokens_path)
    delim = detect_delimiter(tokens_path, encoding=encoding)

    with tokens_path.open("r", encoding=encoding, newline="") as f:
        reader = csv.DictReader(f, delimiter=delim)
        rows = list(reader)

    if not rows:
        raise ValueError(f"Aucun token lu dans {tokens_path}")

    required = [
        "paragraph_ID",
        "sentence_ID",
        "token_ID_within_sentence",
        "token_ID_within_document",
        "word",
        "lemma",
        "POS_tag",
        "dependency_relation",
        "syntactic_head_ID",
    ]
    headers = rows[0].keys()
    missing = [c for c in required if c not in headers]
    if missing:
        raise ValueError(
            f"Colonnes absentes dans {tokens_path}: {missing}. Headers disponibles: {list(headers)}"
        )

    return rows


def read_entities_span(entity_path, encoding="utf-8"):
    entity_path = Path(entity_path)
    delim = detect_delimiter(entity_path, encoding=encoding)

    with entity_path.open("r", encoding=encoding, newline="") as f:
        reader = csv.DictReader(f, delimiter=delim)
        rows = list(reader)

    if not rows:
        return {}

    headers = rows[0].keys()
    if "start_token" not in headers or "end_token" not in headers:
        raise ValueError(
            f"Colonnes start_token / end_token absentes dans {entity_path}. "
            f"Headers disponibles: {list(headers)}"
        )

    entity_map = {}

    for row in rows:
        try:
            start = int((row.get("start_token") or "").strip())
            end   = int((row.get("end_token") or "").strip())

            ent_type = (row.get("cat") or "").strip() or "NaN"
            ent_name = (row.get("COREF") or "").strip()
            if ent_name == "":
                ent_name = (row.get("text") or "").strip()
            if ent_name == "":
                ent_name = "NaN"

            for i in range(start, end + 1):
                entity_map[i] = (ent_name, ent_type)
        except Exception:
            continue

    return entity_map


def load_dynamouv_dict(dynamouv_csv, encoding="utf-8"):
    d = {}
    path = Path(dynamouv_csv)
    if not path.exists():
        return d

    with path.open("r", encoding=encoding, newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        headers = reader.fieldnames or []

        key_col = "Verbe"
        val_col = "Macro-catégorie"   # remplacez par "Catégorie de base" si souhaité

        if key_col not in headers or val_col not in headers:
            raise ValueError(
                f"Colonnes attendues absentes dans {path}. "
                f"Attendu: {key_col=} et {val_col=}. Headers: {headers}"
            )

        for row in reader:
            k = normalize_for_dynamouv((row.get(key_col) or "").strip().strip('"'))
            v = (row.get(val_col) or "").strip().strip('"')
            if k:
                d[k] = v if v else "NaN"

    return d


## 3) Charger Dynamouv

In [4]:
dic_dynamouv = load_dynamouv_dict(dynamouv_csv)
print("Taille dic_dynamouv :", len(dic_dynamouv))
list(dic_dynamouv.items())[:10]


Taille dic_dynamouv : 1067


[('abandonner', 'Dpt au sens large'),
 ('aborder', 'Dynamicité spatiale sans dpt'),
 ('aboutir', 'Dpt au sens large'),
 ('abriver', 'Dynamicité spatiale sans dpt'),
 ('accéder', 'Dpt au sens large'),
 ('accoster', 'Dynamicité spatiale sans dpt'),
 ('accourir', 'Dpt au sens large'),
 ('accrocher', 'Dynamicité spatiale sans dpt'),
 ('achopper', 'Dynamicité spatiale sans dpt'),
 ('alaquer', 'Dynamicité spatiale sans dpt')]

## 4) Reconstruction d’un `.conllu` rempli depuis Propp

Points importants :
- `ID` CoNLL-U = `token_ID_within_sentence + 1`
- `HEAD` CoNLL-U :
  - `0` si `dependency_relation == ROOT`
  - sinon conversion de `syntactic_head_ID` (ID document) en ID local à la phrase
- `UPOS = POS_tag`
- `XPOS = POS_tag`
- `FEATS = _` (car pas de colonne morphologique générale exportée ici)

Les colonnes 9–10 restent vos colonnes custom.


In [5]:
def process_propp_to_conllu(inputDir, outputDir, myNameFile, dic_dynamouv, encoding="utf-8"):
    inputDir = Path(inputDir)
    outputDir = Path(outputDir)

    tokens_path = inputDir / f"{myNameFile}.tokens"
    rows = read_tokens_dict(tokens_path, encoding=encoding)

    entity_path = inputDir / f"{myNameFile}.entities"
    if entity_path.exists():
        try:
            entity_map = read_entities_span(entity_path, encoding=encoding)
        except Exception as e:
            print(f"Erreur dans le fichier {entity_path}: {e}")
            entity_map = {}
    else:
        entity_map = {}

    # Mapping doc_id -> row
    by_doc_id = {}
    for row in rows:
        doc_id = int(row["token_ID_within_document"])
        by_doc_id[doc_id] = row

    conllu_lines = []
    current_sentence = None
    align_id = 0

    for row in rows:
        par_id = row["paragraph_ID"]
        sent_id = row["sentence_ID"]

        if current_sentence != sent_id:
            if current_sentence is not None:
                conllu_lines.append("")
            conllu_lines.append("# Nom fichier = " + myNameFile)
            conllu_lines.append("# Num phrase = " + str(sent_id))
            conllu_lines.append("# Num paragr = " + str(par_id))
            current_sentence = sent_id

        tok_doc_id = int(row["token_ID_within_document"])
        tok_sent_id0 = int(row["token_ID_within_sentence"])   # 0-based dans Propp
        conllu_id = tok_sent_id0 + 1                          # 1-based dans CoNLL-U

        form = row["word"]
        lemma = row["lemma"] if row["lemma"] else "_"
        upos = row["POS_tag"] if row["POS_tag"] else "_"
        xpos = row["POS_tag"] if row["POS_tag"] else "_"
        feats = "_"
        deprel = row["dependency_relation"] if row["dependency_relation"] else "_"

        # Conversion HEAD document -> HEAD local à la phrase
        head_doc_id = int(row["syntactic_head_ID"])
        if deprel.upper() == "ROOT" or head_doc_id == tok_doc_id:
            head = 0
            deprel = "root"
        else:
            if head_doc_id not in by_doc_id:
                head = 0
            else:
                head_row = by_doc_id[head_doc_id]
                # Sécurité : si la tête n'est pas dans la même phrase, on met 0
                if head_row["sentence_ID"] != sent_id:
                    head = 0
                else:
                    head = int(head_row["token_ID_within_sentence"]) + 1

        # Colonnes custom 9-10 : entities > dynamouv > _
        if tok_doc_id in entity_map:
            val_col8, val_col9 = entity_map[tok_doc_id]
        else:
            key = normalize_for_dynamouv(form)
            if key and key in dic_dynamouv:
                val_col8 = dic_dynamouv[key]
                val_col9 = "VERB"
            else:
                val_col8, val_col9 = "_", "_"

        form_align = str(form).strip()

        if upos == "SPACE":
            align_id_token = "_"
        elif form_align in ["-", "–", "—"]:
            align_id_token = "_"
        else:
            form_align = form_align.lstrip("-–—")
            if form_align == "":
                align_id_token = "_"
            else:
                align_id_token = align_id
                align_id += 1

        conllu_lines.append(
            f"{conllu_id}\t{form}\t{lemma}\t{upos}\t{xpos}\t{feats}\t{head}\t{deprel}\t{val_col8}\t{val_col9}\t{tok_doc_id}\t{align_id_token}"
        )

    out_path = outputDir / f"{myNameFile}.conllu"
    with out_path.open("w", encoding=encoding) as f:
        f.write("\n".join(conllu_lines))

    return str(out_path)

## 5) Exécution sur tous les fichiers `.tokens`

In [6]:
tokens_files = sorted([f for f in os.listdir(inputDir) if f.endswith(".tokens")])
print("Nombre de fichiers .tokens:", len(tokens_files))

out_paths = []
for i, myFile in enumerate(tokens_files, 1):
    myNameFile = myFile.replace(".tokens", "")
    print(f"[{i}/{len(tokens_files)}] {myFile}")
    out_paths.append(process_propp_to_conllu(inputDir, outputDir, myNameFile, dic_dynamouv))

print("\n✅ Conversion terminée.")
print("Exemple fichier généré:", out_paths[0] if out_paths else "—")


Nombre de fichiers .tokens: 1
[1/1] tdm_auto_chap1to5.tokens

✅ Conversion terminée.
Exemple fichier généré: ..\results\conllu\from_propp\tdm_auto_chap1to5.conllu


## 6) Contrôle rapide : aperçu d’un `.conllu`

In [7]:
if out_paths:
    sample_path = out_paths[0]
    print("Extrait de:", sample_path, "\n")
    with open(sample_path, "r", encoding="utf-8") as f:
        for _ in range(40):
            line = f.readline()
            if not line:
                break
            print(line.rstrip("\n"))


Extrait de: ..\results\conllu\from_propp\tdm_auto_chap1to5.conllu 

# Nom fichier = tdm_auto_chap1to5
# Num phrase = 0
# Num paragr = 0
1	En	en	ADP	ADP	_	3	case	en l' année 1872	TIME	0	0
2	l'	le	DET	DET	_	3	det	en l' année 1872	TIME	1	1
3	année	année	NOUN	NOUN	_	30	obl:mod	en l' année 1872	TIME	2	2
4	1872	1872	NUM	NUM	_	3	nummod	en l' année 1872	TIME	3	3
5	,	,	PUNCT	PUNCT	_	30	punct	_	_	4	4
6	la	le	DET	DET	_	7	det	la maison	FAC	5	5
7	maison	maison	NOUN	NOUN	_	30	nsubj:pass	la maison	FAC	6	6
8	portant	porter	VERB	VERB	_	7	acl	_	_	7	7
9	le	le	DET	DET	_	10	det	_	_	8	8
10	numéro	numéro	NOUN	NOUN	_	8	obj	_	_	9	9
11	7	7	NUM	NUM	_	10	nummod	_	_	10	10
12	de	de	ADP	ADP	_	10	nmod	_	_	11	11
13	Saville	saville	PROPN	PROPN	_	10	nmod	saville - row	FAC	12	12
14	-	-	PROPN	PROPN	_	10	nmod	saville - row	FAC	13	_
15	row	row	PROPN	PROPN	_	10	nmod	saville - row	FAC	14	13
16	,	,	PUNCT	PUNCT	_	10	punct	_	_	15	14
17	Burlington	burlington	PROPN	PROPN	_	10	appos	burlington gardens	FAC	16	15
18	Gardens	gardens	X

## 7) Contrôle ciblé : vérifier que `HEAD` est bien local à la phrase

On affiche pour une phrase :
- `ID`
- `FORM`
- `HEAD`
- `DEPREL`


In [8]:
if out_paths:
    sample_path = Path(out_paths[0])
    sent_block = []
    with sample_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#") or line == "":
                if sent_block:
                    break
                continue
            sent_block.append(line.split("\t"))

    for row in sent_block:
        print({
            "ID": row[0],
            "FORM": row[1],
            "HEAD": row[6],
            "DEPREL": row[7],
        })


{'ID': '1', 'FORM': 'En', 'HEAD': '3', 'DEPREL': 'case'}
{'ID': '2', 'FORM': "l'", 'HEAD': '3', 'DEPREL': 'det'}
{'ID': '3', 'FORM': 'année', 'HEAD': '30', 'DEPREL': 'obl:mod'}
{'ID': '4', 'FORM': '1872', 'HEAD': '3', 'DEPREL': 'nummod'}
{'ID': '5', 'FORM': ',', 'HEAD': '30', 'DEPREL': 'punct'}
{'ID': '6', 'FORM': 'la', 'HEAD': '7', 'DEPREL': 'det'}
{'ID': '7', 'FORM': 'maison', 'HEAD': '30', 'DEPREL': 'nsubj:pass'}
{'ID': '8', 'FORM': 'portant', 'HEAD': '7', 'DEPREL': 'acl'}
{'ID': '9', 'FORM': 'le', 'HEAD': '10', 'DEPREL': 'det'}
{'ID': '10', 'FORM': 'numéro', 'HEAD': '8', 'DEPREL': 'obj'}
{'ID': '11', 'FORM': '7', 'HEAD': '10', 'DEPREL': 'nummod'}
{'ID': '12', 'FORM': 'de', 'HEAD': '10', 'DEPREL': 'nmod'}
{'ID': '13', 'FORM': 'Saville', 'HEAD': '10', 'DEPREL': 'nmod'}
{'ID': '14', 'FORM': '-', 'HEAD': '10', 'DEPREL': 'nmod'}
{'ID': '15', 'FORM': 'row', 'HEAD': '10', 'DEPREL': 'nmod'}
{'ID': '16', 'FORM': ',', 'HEAD': '10', 'DEPREL': 'punct'}
{'ID': '17', 'FORM': 'Burlington', 'HEAD'